In [ ]:
import torch
import numpy as np
from typing import List,Literal,Type 

In [ ]:
from inference import load_diffusion_model, ddim_reverse_step,reverse_diffusion_ddim, make_ddim_timesteps
from diffusion import DiffusionSchedule,diffuse
from network import UNetDiffusion


In [ ]:
N_STEPS = 1000
T_START = 600
K = 4
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")

In [ ]:
model,schedule = load_diffusion_model(ckpt_path="../models/unet32.pt",device=device)


In [ ]:
patch = np.load("../dataset/test/patch_9107_22.npy").astype(np.float32).squeeze()
patch = (patch - patch.mean()) / (patch.std() + 1e-8)

x0 = torch.from_numpy(patch)[None, None].to(schedule.device)  # (1, 1, 48, 48)
t = torch.tensor([T_START], device=x0.device)

xt, noise = diffuse(x0, t, schedule)

In [ ]:
def _check_steps_inputs(K:int,t_start:int, schedule:DiffusionSchedule):
    if not 0 <= t_start < schedule.timesteps:
        raise ValueError("t_start must be in [0, schedule.timesteps - 1]")
    if not 1 <= K <= t_start + 1:
        raise ValueError("K must be in [1, t_start + 1]")

In [ ]:

def linear_timesteps(K:int,t_start:int, schedule:DiffusionSchedule)->List[int]:
    _check_steps_inputs(K,t_start,schedule)
    linear_steps = make_ddim_timesteps(
        num_ddim_steps=K,
        num_diffusion_steps=schedule.timesteps,
        t_start=t_start,
    )
    return linear_steps

lin_steps = linear_timesteps(K,T_START,schedule)

In [ ]:
def quadratic_timesteps(K:int, schedule:DiffusionSchedule, t_start:int,*, curve:float)->List[int]:
    _check_steps_inputs(K,t_start,schedule)
    n = K-1
    slope = (t_start- curve * n**2)/n
    steps = [round((curve*(x**2))+ slope*x) for x in range(K)] 
    return steps[::-1]

In [ ]:
def uniform_log_snr_steps(K: int, t_start: int, schedule: DiffusionSchedule) -> List[int]:
    """Choose K reverse timesteps evenly spaced in the schedule's log-SNR."""
    _check_steps_inputs(K,t_start,schedule)
    alpha_bar = schedule.alpha_bar[: t_start + 1].detach().to("cpu", dtype=torch.float64)
    log_snr = torch.log(alpha_bar) - torch.log1p(-alpha_bar)
    targets = torch.linspace(
        log_snr[t_start].item(), log_snr[0].item(), K, dtype=torch.float64
    )
    steps = [t_start]
    for i, target in enumerate(targets[1:-1], start=1):
        nearest = int(torch.argmin(torch.abs(log_snr - target)))
        # Reserve one distinct integer timestep for every remaining step.
        lower = K - 1 - i
        upper = steps[-1] - 1
        steps.append(min(max(nearest, lower), upper))
    steps.append(0)
    return steps

log_snr_steps = uniform_log_snr_steps(K, T_START, schedule)
log_snr_steps

In [ ]:
x0_hat, scores = reverse_diffusion_ddim(xt,lin_steps,model,schedule)

In [ ]:
def residual_map(x0:torch.Tensor, x0_hat:torch.Tensor)-> torch.Tensor:
    return torch.abs(x0-x0_hat)/(1+torch.abs(x0_hat)) 

In [ ]:
residual_map(x0,x0_hat)

In [ ]:
def mae_metric(x0:torch.Tensor, x0_hat:torch.Tensor)->float:
    e = residual_map(x0,x0_hat)
    mae = torch.mean(torch.abs(e)).item()
    return mae

In [ ]:
from dataclasses import dataclass
from collections.abc import Callable

StepCurve = Callable[[int, int, DiffusionSchedule], list[int]] # arguments : K, t_start, schedule ; résultat : timesteps décroissants

@dataclass
class PurificationConfig:
    t_start:int
    K:int
    schedule: DiffusionSchedule
    step_curve: StepCurve
    

In [ ]:
def score_config(model:UNetDiffusion, config:PurificationConfig, patch:np.ndarray):
